# 04 — Statistical Analysis and Visualizations

**Purpose:** calculate article-level outcomes by hierarchy level, ROUGE overlap against aligned XSum reference summaries, descriptive transition counts, bootstrap intervals, and per-level Spearman associations.

This stage reads persisted inference and annotation outputs only. Bootstrap resampling is clustered by article because claims from the same article are not independent. Generated charts are actual outputs from the run; no illustrative or fabricated result values are included.

In [ ]:
from pathlib import Path
import sys
import subprocess
import importlib
import pandas as pd
from IPython.display import display, Image

# --------------------------------------------------
# 1. Project paths
# --------------------------------------------------
PROJECT_PARENT = Path("/content/drive/MyDrive/NCM Research")
PROJECT_ROOT = PROJECT_PARENT / "github"
SRC_ROOT = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"

# --------------------------------------------------
# 2. Ensure Google Drive is mounted
# --------------------------------------------------
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# --------------------------------------------------
# 3. Validate repository and inputs
# --------------------------------------------------
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Repository not found: {PROJECT_ROOT}")

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(f"Missing pyproject.toml in {PROJECT_ROOT}")

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Config not found: {CONFIG_PATH}")

# --------------------------------------------------
# 4. Install the missing analysis dependency
# --------------------------------------------------
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "rouge-score>=0.1.2",
    ],
    check=True,
)

# Ensure the repository package is registered
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "--no-build-isolation",
        "-e", str(PROJECT_ROOT),
    ],
    check=True,
)

sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()

# --------------------------------------------------
# 5. Load config and run analysis
# --------------------------------------------------
from ncm_factuality.config import load_config, resolve_paths
from ncm_factuality.analysis import run_analysis

cfg = load_config(CONFIG_PATH)
paths = resolve_paths(PROJECT_ROOT, cfg)
run_dir = Path(paths["run_dir"])

print(f"Project root: {PROJECT_ROOT}")
print(f"Run directory: {run_dir}")
print("Running analysis...")

summary = run_analysis(cfg, run_dir)

print("\nAnalysis summary")
print(summary)

# --------------------------------------------------
# 6. Display level metrics
# --------------------------------------------------
level_metrics_path = run_dir / "level_metrics.csv"

print("\nLevel metrics")
if level_metrics_path.is_file():
    display(pd.read_csv(level_metrics_path))
else:
    print(f"Expected output not found: {level_metrics_path}")

# --------------------------------------------------
# 7. Display transition metrics
# --------------------------------------------------
transition_metrics_path = run_dir / "transition_metrics.csv"

print("\nTransition counts")
if transition_metrics_path.is_file():
    display(pd.read_csv(transition_metrics_path))
else:
    print(f"Expected output not found: {transition_metrics_path}")

# --------------------------------------------------
# 8. Display generated figures
# --------------------------------------------------
figures_dir = run_dir / "figures"
figures = sorted(figures_dir.glob("*.png")) if figures_dir.is_dir() else []

print("\nFigures")
if figures:
    for figure_path in figures:
        print(figure_path.name)
        display(Image(filename=str(figure_path)))
else:
    print(f"No PNG figures found in {figures_dir}")

print("\nAnalysis cell finished.")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/NCM Research/github")
RUN_DIR = (
    Path("/content/drive/MyDrive/NCM Research/local_outputs/runs")
    / "ncm-xsum-factuality-full-test-v1-f672d58824"
)

# A. Find the code that generates the label-distribution plot
analysis_file = PROJECT_ROOT / "src/ncm_factuality/analysis.py"
lines = analysis_file.read_text(encoding="utf-8").splitlines()

print("=" * 70)
print("A. PLOTTING CODE")
print("=" * 70)

terms = ("claim_label_distribution", "Source-evidence labels",
         "claim_fraction_supported", "bar(", "stackplot")

matches = [
    i for i, line in enumerate(lines)
    if any(term.lower() in line.lower() for term in terms)
]

shown = set()
for i in matches:
    start, end = max(0, i - 5), min(len(lines), i + 12)
    if any(j in shown for j in range(start, end)):
        continue
    shown.update(range(start, end))
    print(f"\n--- analysis.py lines {start + 1}-{end} ---")
    for j in range(start, end):
        print(f"{j + 1:4}: {lines[j]}")

# B. Inspect the saved level metrics
print("\n" + "=" * 70)
print("B. LEVEL METRICS")
print("=" * 70)

metrics_file = RUN_DIR / "level_metrics.csv"
if not metrics_file.exists():
    print(f"Missing: {metrics_file}")
else:
    df = pd.read_csv(metrics_file)
    print("Shape:", df.shape)
    print("Columns:")
    for col in df.columns:
        print(" -", col)

    candidates = [
        c for c in df.columns
        if "claim_fraction_" in c and c.endswith("_mean")
    ]
    inspect_cols = ["level_index", "n_articles"] + candidates
    inspect_cols = [c for c in inspect_cols if c in df.columns]

    print("\nRelevant data:")
    display(df[inspect_cols])

# C. Check the generated image files
print("\n" + "=" * 70)
print("C. SAVED FIGURES")
print("=" * 70)

figures_dir = RUN_DIR / "figures"
if figures_dir.exists():
    for p in sorted(figures_dir.glob("*.png")):
        print(f"{p.name}: {p.stat().st_size:,} bytes")
else:
    print("Figures directory does not exist.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Existing experiment output directory
RUN_DIR = (
    Path("/content/drive/MyDrive/NCM Research/local_outputs/runs")
    / "ncm-xsum-factuality-full-test-v1-f672d58824"
)

METRICS_PATH = RUN_DIR / "level_metrics.csv"
FIGURES_DIR = RUN_DIR / "figures"
OUTPUT_PATH = FIGURES_DIR / "claim_label_distribution_by_level.png"

if not METRICS_PATH.is_file():
    raise FileNotFoundError(f"Metrics CSV not found: {METRICS_PATH}")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(METRICS_PATH).sort_values("level_index")

# Plot the per-level mean fractions saved by the analysis stage.
series = {
    "Supported": "claim_fraction_supported_mean",
    "Contradicted": "claim_fraction_contradicted_mean",
    "Unresolved": "claim_fraction_unresolved_mean",
}

required = ["level_index", *series.values()]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in level_metrics.csv: {missing}")

x = df["level_index"].to_numpy(dtype=float)
values = {
    label: pd.to_numeric(df[column], errors="coerce").to_numpy(dtype=float)
    for label, column in series.items()
}

for label, y in values.items():
    if not np.isfinite(y).all():
        raise ValueError(f"{label} contains missing or non-finite values.")
    if ((y < 0) | (y > 1)).any():
        raise ValueError(f"{label} fractions fall outside [0, 1].")

# Confirm the three fractions approximately sum to one at each level.
totals = sum(values.values())
if not np.allclose(totals, 1.0, atol=0.02):
    raise ValueError(
        "Label fractions do not sum approximately to 1 at every level. "
        f"Observed totals: {totals}"
    )

fig, ax = plt.subplots(figsize=(11, 6.5), constrained_layout=True)

for label, y in values.items():
    ax.plot(x, y, marker="o", linewidth=2.4, markersize=6, label=label)

ax.set(
    title="Source-evidence labels by hierarchy level",
    xlabel="NCM hierarchy level (coarse → fine)",
    ylabel="Mean fraction of candidate claims",
    xticks=x,
    ylim=(0, 1),
)
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="best", frameon=False)

fig.savefig(OUTPUT_PATH, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Corrected figure saved to: {OUTPUT_PATH}")
print(f"Plotted hierarchy levels: {len(df)}")
print("Label fractions used:")
display(df[["level_index", *series.values()]])

assert OUTPUT_PATH.is_file() and OUTPUT_PATH.stat().st_size > 0
print("SUCCESS: the corrected figure was saved.")

## Interpretation checklist

- Inspect denominators as well as fractions and intervals.
- Treat automated NLI labels as noisy support evidence, not ground truth.
- ROUGE measures reference overlap; it is not a factuality measure.
- Correlation is descriptive and does not show that one metric causes another.
- Explain failures and unmatched cases; don't cherry-pick levels or figures.